# **A. Langkah Praktikum**

In [ ]:
!pip install -q duckdb beautifulsoup4 lxml requests pyarrow pandas

## 1. H-2.1 — Mengunduh dataset dan memeriksa data

In [ ]:
import os, re, json, time, zlib, queue, threading, collections
from pathlib import Path
import urllib.request
import pandas as pd

URL_DATA = (
    "https://raw.githubusercontent.com/ibamibrahim/"
    "dataset-judul-berita-indonesia/master/detik_news_title.csv"
)

if not os.path.exists("detik_news_title.csv"):
    urllib.request.urlretrieve(URL_DATA, "detik_news_title.csv")

df_sumber = pd.read_csv("detik_news_title.csv")

print("Ukuran berkas :", round(
    os.path.getsize("detik_news_title.csv") / 1e6, 1
), "MB")

print("Baris, kolom :", df_sumber.shape)
print("Kolom        :", list(df_sumber.columns))

Ukuran berkas : 16.1 MB
Baris, kolom : (91017, 4)
Kolom        : ['date', 'url', 'title', 'category']


Kode ini mengunduh dataset judul berita dari GitHub apabila file belum tersedia, kemudian membacanya menggunakan pandas. Informasi ukuran file, jumlah baris dan kolom, serta nama kolom ditampilkan untuk memastikan bahwa dataset berhasil dimuat dengan benar.

## 2. H-2.2 — Membuat portal berita lokal

In [ ]:
import http.server, urllib.parse, html

HALAMAN_UKURAN = 100  # judul per halaman


def jalankan_portal(df, port=8000):
    """Portal berita lokal: menyajikan judul dalam halaman HTML."""
    baris = df.to_dict("records")
    n_hal = (len(baris) + HALAMAN_UKURAN - 1) // HALAMAN_UKURAN

    class Handler(http.server.BaseHTTPRequestHandler):

        def log_message(self, *args):
            # Matikan log
            pass

        def _kirim(self, kode, teks, tipe="text/html; charset=utf-8"):
            data = teks.encode("utf-8")
            self.send_response(kode)
            self.send_header("Content-Type", tipe)
            self.send_header("Content-Length", str(len(data)))
            self.end_headers()
            self.wfile.write(data)

        def do_GET(self):
            u = urllib.parse.urlparse(self.path)

            if u.path == "/robots.txt":
                self._kirim(
                    200,
                    "User-agent: *\nDisallow: /arsip-privat/\n",
                    "text/plain"
                )

            elif u.path == "/arsip-privat/":
                self._kirim(
                    200,
                    "<p>Halaman ini tidak boleh di-crawl.</p>"
                )

            elif u.path == "/berita":
                q = urllib.parse.parse_qs(u.query)
                hal = int(q.get("hal", ["1"])[0])

                if not 1 <= hal <= n_hal:
                    return self._kirim(
                        404, "<p>Halaman tidak ada</p>"
                    )

                potong = baris[
                    (hal - 1) * HALAMAN_UKURAN:
                    hal * HALAMAN_UKURAN
                ]

                isi = []

                for r in potong:
                    iso = pd.to_datetime(
                        r["date"], format="%m/%d/%Y"
                    ).strftime("%Y-%m-%d")

                    isi.append(
                        '<article class="item">'
                        f'<h2 class="judul"><a '
                        f'href="{html.escape(r["url"])}">'
                        f'{html.escape(r["title"])}</a></h2>'
                        f'<span class="kategori">'
                        f'{html.escape(r["category"])}</span>'
                        f'<time datetime="{iso}">'
                        f'{html.escape(r["date"])}</time>'
                        '</article>'
                    )

                self._kirim(
                    200,
                    f"<html><body><h1>Portal Contoh</h1>"
                    f"{''.join(isi)}"
                    f'<nav data-total="{n_hal}"></nav></body></html>'
                )

            else:
                self._kirim(404, "<p>Tidak ditemukan</p>")

    server = http.server.ThreadingHTTPServer(
        ("127.0.0.1", port), Handler
    )

    threading.Thread(
        target=server.serve_forever, daemon=True
    ).start()

    return server, n_hal


server, N_HALAMAN = jalankan_portal(df_sumber)
BASE = "http://127.0.0.1:8000"

print(
    "Portal lokal aktif di", BASE,
    "| jumlah halaman:", N_HALAMAN
)

Portal lokal aktif di http://127.0.0.1:8000 | jumlah halaman: 911


Kode ini membuat server berita lokal yang menyajikan dataset dalam halaman HTML, dengan maksimal 100 judul pada setiap halaman. Server menyediakan jalur /berita, /robots.txt, dan /arsip-privat/, sehingga mahasiswa dapat mempraktikkan crawling serta aturan akses tanpa membebani situs berita sungguhan.

## 3. H-3.1 — Membaca aturan robots.txt

In [ ]:
import requests
import urllib.robotparser
from bs4 import BeautifulSoup

UA = "KelasBigData-UNAND/1.0 (praktikum; kontak: dosen)"


def baca_robots(base):
    r = requests.get(
        base + "/robots.txt",
        headers={"User-Agent": UA},
        timeout=10
    )

    rp = urllib.robotparser.RobotFileParser()
    rp.parse(r.text.splitlines())

    return rp


robots = baca_robots(BASE)

for jalur in ["/berita?hal=1", "/arsip-privat/"]:
    boleh = robots.can_fetch(UA, BASE + jalur)
    print(f"{jalur:<18} boleh di-crawl? {boleh}")

/berita?hal=1      boleh di-crawl? True
/arsip-privat/     boleh di-crawl? False


Kode ini membaca robots.txt dari portal lokal dan memeriksa apakah alamat tertentu boleh diakses oleh crawler. Halaman berita diizinkan untuk diambil, sedangkan /arsip-privat/ dilarang. Pemeriksaan tersebut menjadi bagian dari etika crawling agar program mengikuti aturan akses situs.

## 4. H-3.2 — Parsing halaman HTML dan mengambil judul berita

In [ ]:
def parse_halaman(teks_html):
    """Ubah HTML satu halaman menjadi daftar dict."""
    sup = BeautifulSoup(teks_html, "html.parser")
    hasil = []

    for a in sup.select("article.item"):
        link = a.select_one("h2.judul a")

        hasil.append({
            "judul": link.get_text(strip=True),
            "url": link["href"],
            "kategori": a.select_one(
                "span.kategori"
            ).get_text(strip=True),
            "tanggal": a.select_one("time")["datetime"],
        })

    return hasil

Fungsi parse_halaman() mengubah HTML menjadi daftar dictionary yang berisi judul, URL, kategori, dan tanggal berita. CSS selector digunakan untuk menemukan elemen artikel, tautan judul, kategori, dan tanggal sehingga data yang dibutuhkan dapat diambil tanpa menyimpan seluruh kode HTML.

## 5. H-3.3 — Crawling dengan jeda dan percobaan ulang

In [ ]:
def crawl_iter(base, halaman, delay=0.0, ua=UA):
    """Ambil halaman satu per satu dengan pemeriksaan robots.txt."""
    rp = baca_robots(base)

    sesi = requests.Session()
    sesi.headers["User-Agent"] = ua

    for h in halaman:
        url = f"{base}/berita?hal={h}"

        if not rp.can_fetch(ua, url):
            print("Dilewati (robots.txt):", url)
            continue

        for percobaan in range(3):
            try:
                r = sesi.get(url, timeout=10)
                r.raise_for_status()
                break

            except requests.RequestException:
                time.sleep(0.5 * (percobaan + 1))

        else:
            print("Gagal setelah 3 percobaan:", url)
            continue

        yield from parse_halaman(r.text)
        time.sleep(delay)


contoh = list(crawl_iter(BASE, range(1, 4)))

print("Judul dari 3 halaman:", len(contoh))

for c in contoh[:3]:
    print(c)

Judul dari 3 halaman: 300
{'judul': 'Kemnaker Awasi TKA di Meikarta', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916114/kemnaker-awasi-tka-di-meikarta', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'BNI Digitalkan BNI Java Jazz 2020', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916199/bni-digitalkan-bni-java-jazz-2020', 'kategori': 'finance', 'tanggal': '2020-02-26'}
{'judul': 'Terbang ke Australia, Edhy Prabowo Mau Genjot Budi Daya Lobster', 'url': 'https://finance.detik.com/berita-ekonomi-bisnis/d-4916193/terbang-ke-australia-edhy-prabowo-mau-genjot-budi-daya-lobster', 'kategori': 'finance', 'tanggal': '2020-02-26'}


Fungsi crawl_iter() mengambil halaman berita secara berurutan, memeriksa izin dari robots.txt, dan mencoba kembali permintaan yang gagal hingga tiga kali. Data hasil parsing dikembalikan satu per satu menggunakan yield from. Contoh pengujian mengambil tiga halaman, dengan hasil yang diharapkan sebanyak 300 judul.

## 6. H-4.1 — Membuat parser sitemap berita

In [ ]:
from bs4 import BeautifulSoup
from urllib.parse import urlparse

POLA_ARTIKEL = re.compile(
    r"https?://([\w-]+)\.detik\.com/.+/d-(\d+)/"
)

def parse_sitemap_news(xml_data):
    """Ubah sitemap berita XML menjadi daftar dict."""
    sup = BeautifulSoup(xml_data, "xml")
    hasil = []

    for u in sup.find_all("url"):
        loc, judul, waktu = (
            u.find("loc"),
            u.find("title"),
            u.find("publication_date")
        )

        if loc is None or judul is None:
            continue

        url = loc.get_text(strip=True)
        m = POLA_ARTIKEL.match(url)
        kunci = u.find("keywords")

        hasil.append({
            "judul": re.sub(
                r"\s+", " ", judul.get_text(strip=True)
            ),
            "url": url,
            "kategori": m.group(1) if m else None,
            "id": int(m.group(2)) if m else None,
            "tanggal": (
                waktu.get_text(strip=True)[:10]
                if waktu else None
            ),
            "kata_kunci": (
                kunci.get_text(strip=True)
                if kunci else ""
            ),
        })

    return hasil

Fungsi ini membaca sitemap berformat XML dan mengambil informasi judul, URL, kategori, ID berita, tanggal terbit, serta kata kunci. Ekspresi reguler digunakan untuk mengenali kategori dan ID dari URL.

## 7. H4.2 — Membuat contoh XML untuk menguji parser

In [ ]:
# Contoh XML yang digunakan untuk menguji parser sitemap.
CONTOH_XML = """<?xml version="1.0" encoding="UTF-8"?>
<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9"
        xmlns:news="http://www.google.com/schemas/sitemap-news/0.9">
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729282/mk-kepala-daerah-terdakwa-politik-tak-bisa-diberhentikan-tanpa-usulan-dprd]]></loc>
    <news:news>
      <news:publication>
        <news:name>detikNews</news:name>
        <news:language>id</news:language>
      </news:publication>
      <news:publication_date>2026-10-29T12:52:00+00:00</news:publication_date>
      <news:title><![CDATA[MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD]]></news:title>
      <news:keywords><![CDATA[mahkamah konstitusi,mk,hukum,kepala daerah]]></news:keywords>
    </news:news>
  </url>
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729280/tep-unair-periksa-1-000-sapi-di-manokwari-lewat-posyandu-ternak-keliling]]></loc>
    <news:news>
      <news:publication_date>2026-10-29T12:50:00+00:00</news:publication_date>
      <news:title><![CDATA[TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling]]></news:title>
    </news:news>
  </url>
  <url>
    <loc><![CDATA[https://news.detik.com/berita/d-8729264/andre-rosiade-dorong-sedimen-sungai-akibat-banjir-jadi-sumber-pendapatan-padang]]></loc>
    <news:news>
      <news:publication_date>2026-10-29T12:44:00+00:00</news:publication_date>
      <news:title><![CDATA[Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang]]></news:title>
    </news:news>
  </url>
</urlset>"""

for x in parse_sitemap_news(CONTOH_XML):
    print({
        k: x[k]
        for k in ("judul", "kategori", "id", "tanggal")
    })

print(
    "kata kunci entri pertama:",
    parse_sitemap_news(CONTOH_XML)[0]["kata_kunci"]
)

print(
    "Sama bila masukan berupa bytes:",
    parse_sitemap_news(CONTOH_XML.encode("utf-8"))
    == parse_sitemap_news(CONTOH_XML)
)

{'judul': "MK: Kepala Daerah 'Terdakwa Politik' Tak Bisa Diberhentikan Tanpa Usulan DPRD", 'kategori': 'news', 'id': 8729282, 'tanggal': '2026-10-29'}
{'judul': 'TEP UNAIR Periksa 1.000 Sapi di Manokwari Lewat Posyandu Ternak Keliling', 'kategori': 'news', 'id': 8729280, 'tanggal': '2026-10-29'}
{'judul': 'Andre Rosiade Dorong Sedimen Sungai Akibat Banjir Jadi Sumber Pendapatan Padang', 'kategori': 'news', 'id': 8729264, 'tanggal': '2026-10-29'}
kata kunci entri pertama: mahkamah konstitusi,mk,hukum,kepala daerah
Sama bila masukan berupa bytes: True


Sel ini menyediakan tiga contoh entri sitemap sebagai data uji lokal, sehingga fungsi parser dapat diuji tanpa mengunduh XML dari internet. Pengujian mencetak judul, kategori, ID, tanggal, dan kata kunci, kemudian memeriksa apakah hasil parsing sama ketika input diberikan sebagai teks maupun bytes. Parser pada sel sebelumnya harus dilengkapi terlebih dahulu agar pengujian berhasil.

## 8. H-4.3 — Menentukan aturan crawling sitemap

In [1]:
import urllib.robotparser

UA_DETIK = (
    "BigData/1.0 (praktikum pembelajaran; "
    "kontak: alramadhanny@gmail.com)"
)

DELAY = 2.0
BATAS_PER_BAGIAN = 7
BATAS_TOTAL = 20

SITEMAP_DIPILIH = [
    "https://news.detik.com/berita/sitemap_news.xml",
    "https://inet.detik.com/news/sitemap_news.xml",
    "https://sport.detik.com/basket/sitemap_news.xml",
]

_robots = {}


def boleh_diambil(url, ua=UA_DETIK):
    """Memeriksa robots.txt berdasarkan host."""
    host = urlparse(url).netloc

    if host not in _robots:
        rp = urllib.robotparser.RobotFileParser()
        rp.set_url(f"https://{host}/robots.txt")
        rp.read()
        _robots[host] = rp

    return _robots[host].can_fetch(ua, url)


def ambil_xml(url, sesi, delay=DELAY):
    if not boleh_diambil(url):
        raise PermissionError("robots.txt melarang: " + url)

    r = sesi.get(url, timeout=20)

    if r.status_code in (403, 429):
        raise RuntimeError(
            f"Server membalas {r.status_code}. "
            "Berhenti, tunggu, dan kurangi laju."
        )

    r.raise_for_status()
    time.sleep(delay)

    return r.content


def crawl_detik():
    sesi = requests.Session()
    sesi.headers["User-Agent"] = UA_DETIK

    hasil = []

    for sm in SITEMAP_DIPILIH:
        if len(hasil) >= BATAS_TOTAL:
            break

        hasil += parse_sitemap_news(
            ambil_xml(sm, sesi)
        )[:BATAS_PER_BAGIAN]

    return hasil[:BATAS_TOTAL]

Kode ini membatasi crawling maksimal 20 judul dari tiga sitemap, dengan maksimal tujuh judul dari setiap sitemap. Program memeriksa robots.txt untuk setiap host, memberi jeda dua detik antarpengambilan sitemap, dan berhenti ketika server mengembalikan status 403 atau 429.

## 9. H-4.4 — Menjalankan crawling dan memeriksa hasil

In [ ]:
try:
    sampel = crawl_detik()

except Exception as e:
    sampel = []
    print("Crawling gagal:", type(e).__name__, "-", e)

if sampel:
    Path("contoh_crawl").mkdir(exist_ok=True)

    with open(
        "contoh_crawl/detik_sitemap.jsonl",
        "w",
        encoding="utf-8"
    ) as f:
        for x in sampel:
            f.write(
                json.dumps(x, ensure_ascii=False) + "\n"
            )

    cek = pd.DataFrame(sampel)

    print("Judul terkumpul   :", len(cek))
    print(
        "Judul kosong      :",
        (cek["judul"].str.strip() == "").sum()
    )
    print("URL duplikat      :", cek["url"].duplicated().sum())
    print("Tanggal kosong    :", cek["tanggal"].isna().sum())
    print("Kategori          :", cek["kategori"].value_counts().to_dict())

    print(
        cek[["judul", "kategori", "tanggal"]]
        .head(3)
        .to_string(index=False)
    )

Judul terkumpul   : 20
Judul kosong      : 0
URL duplikat      : 0
Tanggal kosong    : 0
Kategori          : {'news': 7, 'inet': 7, 'sport': 6}
                                                              judul kategori    tanggal
 Pemotor Tewas Terbentur Truk Usai Senggolan di Batuceper Tangerang     news 2026-10-29
      Gas Bocor dari Pabrik di Karawang, Warga Mengeluh Sesak Napas     news 2026-10-29
Pilu Balita di Sulsel Meninggal Usai 10 Hari Tak Makan karena Sakit     news 2026-10-29


Kode ini menjalankan crawler, menyimpan hasil pengambilan dalam format JSONL di folder contoh_crawl, kemudian memeriksa jumlah judul, judul kosong, URL duplikat, tanggal kosong, dan distribusi kategori. Modul mengharapkan 20 judul, tetapi isi berita dapat berubah sesuai kondisi sitemap saat kode dijalankan.

## 10. H-5.1 — Menyiapkan antrean dan statistik pipeline

In [ ]:
RAW = Path("lapisan_mentah")
RAW.mkdir(exist_ok=True)

for f in RAW.glob("*.jsonl"):
    f.unlink()

channel = queue.Queue(maxsize=2000)
STOP = object()

statistik = {
    "max_antrean": 0,
    "masuk": 0,
    "keluar": 0,
    "file": 0
}

Sel ini menyiapkan folder lapisan_mentah untuk menyimpan data mentah dan menghapus file JSONL lama yang ada di dalamnya. queue.Queue(maxsize=2000) berfungsi sebagai channel berkapasitas terbatas, sedangkan variabel statistik mencatat jumlah data masuk, jumlah data keluar, ukuran antrean maksimum, dan jumlah file yang ditulis.

## 11. H-5.2 — Membuat source dan sink

In [ ]:
def source():
    for ev in crawl_iter(BASE, range(1, N_HALAMAN + 1)):
        channel.put(ev)
        statistik["masuk"] += 1
        statistik["max_antrean"] = max(
            statistik["max_antrean"], channel.qsize()
        )

    channel.put(STOP)


def sink(ukuran_batch=5000):
    batch, nomor = [], 0

    def tulis():
        nonlocal batch, nomor

        nomor += 1

        with open(
            RAW / f"batch_{nomor:04d}.jsonl",
            "w",
            encoding="utf-8"
        ) as f:
            for ev in batch:
                f.write(
                    json.dumps(ev, ensure_ascii=False) + "\n"
                )

        statistik["file"] = nomor
        batch = []

    while True:
        ev = channel.get()

        if ev is STOP:
            break

        batch.append(ev)
        statistik["keluar"] += 1

        if len(batch) >= ukuran_batch:
            tulis()

    if batch:
        tulis()

Fungsi source() mengambil berita dari seluruh halaman portal lokal dan memasukkannya ke dalam antrean. Fungsi sink() mengambil data dari antrean lalu menuliskannya ke file JSONL setiap 5.000 data. Objek STOP menandakan bahwa pengiriman data telah selesai, sedangkan antrean terbatas membuat source menunggu ketika sink belum mampu memproses data secepat pengiriman.

## 12. H-5.3 — Menjalankan pipeline dan memeriksa statistik

In [ ]:
t0 = time.perf_counter()

t_src = threading.Thread(target=source)
t_snk = threading.Thread(target=sink)

t_src.start()
t_snk.start()

t_src.join()
t_snk.join()

waktu = time.perf_counter() - t0

print("Event masuk channel :", f"{statistik['masuk']:,}")
print("Event ditulis sink :", f"{statistik['keluar']:,}")

print(
    "File batch          :",
    statistik["file"],
    "->",
    sorted(p.name for p in RAW.glob("*.jsonl"))[:3],
    "..."
)

print(
    "Antrean maks <= 2000:",
    statistik["max_antrean"] <= 2000
)

print(
    "Total ukuran mentah :",
    round(
        sum(p.stat().st_size for p in RAW.glob("*.jsonl"))
        / 1e6,
        1
    ),
    "MB"
)

print(
    "Waktu crawl total   :",
    round(waktu, 1),
    "detik untuk",
    N_HALAMAN,
    "halaman"
)

Event masuk channel : 91,017
Event ditulis sink : 91,017
File batch          : 19 -> ['batch_0001.jsonl', 'batch_0002.jsonl', 'batch_0003.jsonl'] ...
Antrean maks <= 2000: True
Total ukuran mentah : 20.7 MB
Waktu crawl total   : 60.8 detik untuk 911 halaman


Kode ini menjalankan source dan sink pada thread terpisah, lalu mengukur durasi pipeline dan menampilkan statistik hasilnya. Berdasarkan contoh modul, 91.017 event masuk harus sama dengan 91.017 event yang ditulis. Kesamaan tersebut menjadi pemeriksaan awal untuk mendeteksi kemungkinan kehilangan data selama proses ingestion.

## 13. Membaca data mentah dan membuat aturan kualitas

In [ ]:
KATEGORI_DIKENAL = {
    "finance", "food", "health", "hot", "inet",
    "news", "oto", "sport", "travel"
}

def aturan_kualitas(d):
    """Setiap aturan menghasilkan Series True/False per baris."""
    jml_kata = d["judul"].fillna("").str.split().str.len()

    return {
        "judul tidak kosong":
            d["judul"].fillna("").str.strip() != "",

        "judul 3-30 kata":
            jml_kata.between(3, 30),

        "kategori dikenal":
            d["kategori"].isin(KATEGORI_DIKENAL),

        "tanggal valid":
            pd.to_datetime(
                d["tanggal"],
                format="%Y-%m-%d",
                errors="coerce"
            ).notna(),

        "url artikel valid":
            d["url"].fillna("").str.contains(
                r"/d-\d+", regex=True
            ),

        "judul tidak duplikat":
            ~d["judul"].duplicated(keep=False),
    }

Kode ini membaca semua file JSONL dari lapisan_mentah, menggabungkannya menjadi satu DataFrame, lalu memeriksa kualitas setiap baris berdasarkan kelengkapan judul, jumlah kata, kategori, validitas tanggal dan URL, serta duplikasi judul.

## 14. H-6.2 — Membersihkan data

In [ ]:
bersih = mentah.copy()

bersih["judul"] = (
    bersih["judul"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

bersih = bersih.drop_duplicates(
    subset="judul",
    keep="first"
)

bersih = bersih[
    bersih["judul"].str.split().str.len().between(3, 30)
].copy()

bersih["tanggal"] = pd.to_datetime(
    bersih["tanggal"],
    format="%Y-%m-%d"
)

bersih["bulan"] = bersih["tanggal"].dt.strftime("%Y-%m")

bersih["id"] = (
    bersih["url"]
    .str.extract(r"/d-(\d+)")[0]
    .astype("Int64")
)

bersih["url"] = bersih["url"].where(
    bersih["url"].str.contains(r"/d-\d+", regex=True)
)

print(
    "Baris setelah dibersihkan:",
    f"{len(bersih):,}",
    "| dibuang:",
    f"{len(mentah) - len(bersih):,}"
)

Baris setelah dibersihkan: 90,818 | dibuang: 199


Kode ini membersihkan spasi berlebih, menghapus judul duplikat, mempertahankan judul dengan panjang 3–30 kata, mengubah tanggal menjadi tipe datetime, serta membuat kolom bulan dan ID berita. URL yang tidak memiliki pola ID artikel akan dikosongkan, tetapi barisnya tetap dipertahankan agar judul dari kategori inet yang URL-nya rusak tidak langsung hilang dari analisis.

## 15. H-6.3 — Membandingkan pilihan partisi Parquet

In [ ]:
import shutil
import pyarrow.dataset as ds


def simpan_terpartisi(data, kolom_partisi, folder):
    shutil.rmtree(folder, ignore_errors=True)

    data.to_parquet(
        folder,
        partition_cols=kolom_partisi,
        index=False
    )

    n = len(list(Path(folder).rglob("*.parquet")))

    ukuran = (
        sum(
            p.stat().st_size
            for p in Path(folder).rglob("*.parquet")
        ) / 1e6
    )

    return n, ukuran


# Bandingkan pilihan kolom partisi.
for kolom in (
    ["kategori"],
    ["bulan"],
    ["kategori", "bulan"]
):
    n, mb = simpan_terpartisi(
        bersih, kolom, "uji_partisi"
    )

    print(
        f"partisi {str(kolom):<22} "
        f"-> {n:>3} file, {mb:5.1f} MB"
    )

shutil.rmtree("uji_partisi")

partisi ['kategori']           ->   9 file,   8.4 MB
partisi ['bulan']              ->   6 file,   8.3 MB
partisi ['kategori', 'bulan']  ->  54 file,   8.4 MB


Fungsi simpan_terpartisi() menyimpan DataFrame sebagai Parquet berdasarkan kolom partisi yang dipilih, kemudian menghitung jumlah file dan total ukurannya. Sel ini membandingkan tiga pilihan partisi agar dapat diketahui struktur penyimpanan yang sesuai untuk analisis berikutnya dan tidak menghasilkan terlalu banyak file kecil.

## 16. H-6.4 — Menyimpan partisi final dan menguji partition pruning

In [ ]:
TERKURASI = "lapisan_terkurasi"

n, mb = simpan_terpartisi(
    bersih,
    ["kategori"],
    TERKURASI
)

print(
    "\nDisimpan ke",
    TERKURASI,
    "dengan partisi kategori:",
    n,
    "file,",
    round(mb, 1),
    "MB"
)

dataset = ds.dataset(
    TERKURASI,
    format="parquet",
    partitioning="hive"
)

semua = list(dataset.get_fragments())

hanya = list(
    dataset.get_fragments(
        filter=ds.field("kategori") == "finance"
    )
)

print(
    "Partition pruning: baca",
    len(hanya),
    "dari",
    len(semua),
    "file untuk kategori = finance"
)


Disimpan ke lapisan_terkurasi dengan partisi kategori: 9 file, 8.4 MB
Partition pruning: baca 1 dari 9 file untuk kategori = finance


Kode ini menyimpan data bersih ke folder lapisan_terkurasi menggunakan partisi berdasarkan kategori. Selanjutnya, PyArrow memeriksa penerapan partition pruning, yaitu pemilihan partisi yang relevan dengan filter. Berdasarkan contoh modul, pencarian kategori finance hanya memerlukan satu dari sembilan file partisi.

## 17. H-7.1 — Membaca Parquet dan melakukan query SQL

In [ ]:
import duckdb

con = duckdb.connect()

con.sql(
    f"""
    CREATE VIEW berita AS
    SELECT *
    FROM read_parquet(
        '{TERKURASI}/*/*.parquet',
        hive_partitioning=true
    )
    """
)

print(
    con.sql("""
        SELECT kategori, COUNT(*) AS jumlah
        FROM berita
        GROUP BY kategori
        ORDER BY jumlah DESC
    """).df().to_string(index=False)
)

print()

print(
    con.sql("""
        SELECT bulan, COUNT(*) AS judul_corona
        FROM berita
        WHERE lower(judul) LIKE '%corona%'
        GROUP BY bulan
        ORDER BY bulan
    """).df().to_string(index=False)
)

kategori  jumlah
    news   32321
     hot   16290
 finance   14141
  travel    6453
    inet    5577
  health    4914
     oto    4375
    food    4314
   sport    2433

  bulan  judul_corona
2020-01           385
2020-02          1077
2020-03          6994
2020-04          4913
2020-05          2934
2020-06           943


Kode ini membuat koneksi DuckDB dan view bernama berita dari file Parquet yang sudah dipartisi. Query pertama menghitung jumlah judul per kategori, sedangkan query kedua menghitung judul yang mengandung kata corona per bulan. Hasilnya digunakan untuk memahami distribusi berita dan perubahan perhatian terhadap topik tersebut sepanjang periode dataset.

## 18. H-7.2 — Mengubah satu baris menjadi dokumen

In [ ]:
dokumen = {
    "id": int(bersih.iloc[0]["id"]),
    "judul": bersih.iloc[0]["judul"],
    "kategori": bersih.iloc[0]["kategori"],
    "tanggal": str(
        bersih.iloc[0]["tanggal"].date()
    ),
    "kata": re.findall(
        r"[a-z]+",
        bersih.iloc[0]["judul"].lower()
    ),
}

print(
    json.dumps(
        dokumen,
        ensure_ascii=False,
        indent=2
    )
)

{
  "id": 4916114,
  "judul": "Kemnaker Awasi TKA di Meikarta",
  "kategori": "finance",
  "tanggal": "2020-02-26",
  "kata": [
    "kemnaker",
    "awasi",
    "tka",
    "di",
    "meikarta"
  ]
}


Kode ini mengambil satu baris pertama dari DataFrame bersih dan mengubahnya menjadi struktur dokumen dengan atribut ID, judul, kategori, tanggal, serta daftar kata dari judul. Format JSON tersebut memperlihatkan bagaimana satu berita dapat disimpan bersama atribut dan hasil pengolahan teksnya dalam satu objek.

## 19. H-8.1 — Menyiapkan data dan fungsi MapReduce

In [ ]:
kasir = [
    [
        "nasi goreng,1",
        "es teh,2",
        "mie ayam,1",
        "es teh,1"
    ],
    [
        "nasi goreng,2",
        "es teh,1",
        "es teh,3",
        "mie ayam,1"
    ],
    [
        "mie ayam,2",
        "nasi goreng,1",
        "es teh,1",
        "nasi goreng,1"
    ],
]


def map_fn(baris):
    menu, jumlah = baris.split(",")
    return (menu, int(jumlah))


def reduce_fn(menu, daftar_nilai):
    return (menu, sum(daftar_nilai))

Sel ini menyiapkan data transaksi dari tiga kasir serta dua fungsi dasar MapReduce. map_fn() memecah setiap transaksi menjadi pasangan menu dan jumlah, sementara reduce_fn() menjumlahkan seluruh nilai yang telah dikelompokkan untuk menu yang sama.

## 20. H-8.2 — Tahap map, shuffle, dan reduce

In [ ]:
# 1. MAP: setiap kasir memproses datanya sendiri.
hasil_map = [
    [map_fn(b) for b in k]
    for k in kasir
]

for nama, h in zip("ABC", hasil_map):
    print("map kasir", nama, "->", h)


# 2. SHUFFLE: kumpulkan nilai berdasarkan key.
shuffle = {}

for h in hasil_map:
    for menu, jumlah in h:
        shuffle.setdefault(menu, []).append(jumlah)

print("\nshuffle:", shuffle)


# 3. REDUCE: gabungkan nilai per key.
hasil = dict(
    reduce_fn(m, v)
    for m, v in shuffle.items()
)

print(
    "\nreduce:",
    hasil,
    "| total porsi:",
    sum(hasil.values())
)

map kasir A -> [('nasi goreng', 1), ('es teh', 2), ('mie ayam', 1), ('es teh', 1)]
map kasir B -> [('nasi goreng', 2), ('es teh', 1), ('es teh', 3), ('mie ayam', 1)]
map kasir C -> [('mie ayam', 2), ('nasi goreng', 1), ('es teh', 1), ('nasi goreng', 1)]

shuffle: {'nasi goreng': [1, 2, 1, 1], 'es teh': [2, 1, 1, 3, 1], 'mie ayam': [1, 1, 2]}

reduce: {'nasi goreng': 5, 'es teh': 8, 'mie ayam': 4} | total porsi: 17


Kode ini menjalankan tiga tahap MapReduce secara berurutan. Pada tahap map, data setiap kasir diubah menjadi pasangan menu dan jumlah; pada shuffle, pasangan dikelompokkan berdasarkan menu; dan pada reduce, seluruh jumlah per menu dijumlahkan. Hasil yang diharapkan adalah nasi goreng 5 porsi, es teh 8 porsi, mie ayam 4 porsi, dengan total 17 porsi.

## 21. H-8.3 — Menggunakan combiner

In [ ]:
def combiner(pasangan):
    c = collections.Counter()

    for menu, jumlah in pasangan:
        c[menu] += jumlah

    return list(c.items())


setelah = [
    combiner(h)
    for h in hasil_map
]

print(
    "\nPasangan dikirim tanpa combiner:",
    sum(len(h) for h in hasil_map)
)

print(
    "Pasangan dikirim dengan combiner:",
    sum(len(h) for h in setelah)
)


Pasangan dikirim tanpa combiner: 12
Pasangan dikirim dengan combiner: 9


Fungsi combiner() menjumlahkan data untuk setiap menu di sisi masing-masing kasir sebelum dikirim ke proses penggabungan berikutnya. Dengan cara ini, jumlah pasangan yang dipindahkan turun dari 12 menjadi 9 pada contoh kantin, tetapi hasil total porsi setiap menu tetap sama.

## 22. H-9.1 — Menentukan stopword dan fungsi tokenisasi

In [ ]:
STOPWORD = set("""
yang di dan ke dari ini itu untuk dengan pada adalah akan tak tidak jadi soal
saat ada bisa juga oleh atau karena sebagai dalam usai hingga jika saja para lebih hanya
sudah belum masih kini lagi pun serta agar bakal dua tiga kok nih ya yg dgn
""".split())


def tokenisasi(teks):
    return re.findall(r"[a-z]+", teks.lower())

Kode ini menetapkan daftar stopword, yaitu kata-kata umum yang dapat dihilangkan agar analisis lebih berfokus pada kata yang bermakna. Fungsi tokenisasi() mengubah teks menjadi huruf kecil, kemudian mengambil rangkaian huruf untuk menghasilkan daftar kata.

## 23. H-9.2 — Menentukan input split dan menjalankan mapper

In [ ]:
# Satu file partisi Parquet menjadi satu input split.
splits = sorted(
    Path(TERKURASI).rglob("*.parquet")
)

print("Jumlah input split:", len(splits))

for p in splits:
    n = len(
        pd.read_parquet(
            p,
            columns=["judul"]
        )
    )

    print(f" {p.parent.name:<16} {n:>6,} judul")


def mapper_split(path):
    """Satu mapper menghasilkan pasangan (kata, 1)."""
    judul = pd.read_parquet(
        path,
        columns=["judul"]
    )["judul"]

    return [
        (kata, 1)
        for t in judul
        for kata in tokenisasi(t)
    ]


hasil_map = [
    mapper_split(p)
    for p in splits
]

total_pasangan = sum(
    len(h)
    for h in hasil_map
)

Jumlah input split: 9
 kategori=finance 14,141 judul
 kategori=food     4,314 judul
 kategori=health   4,914 judul
 kategori=hot     16,290 judul
 kategori=inet     5,577 judul
 kategori=news    32,321 judul
 kategori=oto      4,375 judul
 kategori=sport    2,433 judul
 kategori=travel   6,453 judul


Sel ini menggunakan setiap file partisi Parquet sebagai input split yang diproses oleh mapper. Fungsi mapper_split() membaca kolom judul dan menghasilkan pasangan (kata, 1) untuk setiap kata yang ditemukan. Jumlah split pada contoh modul adalah sembilan, mengikuti sembilan kategori berita.

## 24. H-9.3 — Shuffle, reduce, dan menghitung frekuensi kata

In [ ]:
shuffle = collections.defaultdict(list)

for h in hasil_map:
    for kata, n in h:
        shuffle[kata].append(n)

hasil_semua = {
    k: sum(v)
    for k, v in shuffle.items()
}

print(
    "\nPasangan dari semua mapper:",
    f"{total_pasangan:,}"
)

print(
    "Kata unik (key setelah shuffle):",
    f"{len(hasil_semua):,}"
)

print("\n10 kata teratas (semua kata):")

for k, n in collections.Counter(
    hasil_semua
).most_common(10):
    print(f" {k:<10} {n:>7,}")

top = [
    (k, n)
    for k, n in collections.Counter(
        hasil_semua
    ).most_common()
    if k not in STOPWORD
]

print("\n10 kata teratas (tanpa stopword):")

for k, n in top[:10]:
    print(f" {k:<10} {n:>7,}")


Pasangan dari semua mapper: 825,601
Kata unik (key setelah shuffle): 32,275

10 kata teratas (semua kata):
 di          27,390
 corona      17,239
 ini         10,124
 yang         8,668
 dan          6,785
 ke           5,592
 tak          5,096
 dari         4,697
 jadi         4,486
 untuk        3,723

10 kata teratas (tanpa stopword):
 corona      17,239
 rp           3,154
 virus        3,075
 kasus        2,743
 polisi       2,732
 positif      2,682
 indonesia    2,504
 warga        2,495
 rumah        2,465
 jokowi       2,317


Kode ini mengelompokkan seluruh pasangan berdasarkan kata pada tahap shuffle, lalu menjumlahkan nilai setiap kata pada tahap reduce. Setelah frekuensi dihitung, program menampilkan sepuluh kata teratas sebelum dan setelah stopword dihilangkan untuk membandingkan hasil analisis teks.

## 25. H-9.4 — Membandingkan hasil MapReduce dengan Counter

In [ ]:
cek = collections.Counter(
    kata
    for t in bersih["judul"]
    for kata in tokenisasi(t)
)

print(
    "\nSama dengan Counter biasa:",
    dict(cek) == hasil_semua
)


Sama dengan Counter biasa: True


Sel ini memverifikasi hasil MapReduce dengan perhitungan frekuensi menggunakan collections.Counter secara langsung. Jika output menunjukkan True, hasil kedua metode sama sehingga perhitungan kata melalui tahapan map, shuffle, dan reduce sesuai dengan perhitungan biasa.

## 26. H-10.1 — Membuat mapper yang menggunakan combiner

In [ ]:
from multiprocessing import Pool


def mapper_combiner(path):
    """Mapper dengan combiner berupa Counter lokal."""
    judul = pd.read_parquet(
        path,
        columns=["judul"]
    )["judul"]

    c = collections.Counter()

    for t in judul:
        c.update(tokenisasi(t))

    return c


def jalankan(fn, daftar, proses):
    t0 = time.perf_counter()

    if proses == 1:
        hasil = [fn(x) for x in daftar]

    else:
        with Pool(proses) as pool:
            hasil = pool.map(fn, daftar)

    return hasil, time.perf_counter() - t0

Kode ini membuat mapper yang sekaligus menghitung frekuensi kata lokal menggunakan Counter. Fungsi jalankan() digunakan untuk mengukur waktu pemrosesan dengan satu atau beberapa proses, sehingga hasil mapper biasa dapat dibandingkan dengan mapper yang menggunakan combiner.

## 27. H-10.2 — Mengukur waktu dan jumlah pasangan

In [ ]:
print("CPU tersedia:", os.cpu_count())

print(
    f"{'skenario':<28}"
    f"{'waktu (s)':>10}"
    f"{'pasangan dikirim':>20}"
)

for proses in (1, 2):
    waktu = []

    for _ in range(3):
        h, w = jalankan(
            mapper_split,
            splits,
            proses
        )
        waktu.append(w)

    print(
        f"{('1 proses' if proses == 1 else '2 proses') + ', tanpa combiner':<28}"
        f"{sorted(waktu)[1]:>10.3f}"
        f"{sum(len(x) for x in h):>20,}"
    )

    waktu = []

    for _ in range(3):
        h2, w = jalankan(
            mapper_combiner,
            splits,
            proses
        )
        waktu.append(w)

    print(
        f"{('1 proses' if proses == 1 else '2 proses') + ', dengan combiner':<28}"
        f"{sorted(waktu)[1]:>10.3f}"
        f"{sum(len(x) for x in h2):>20,}"
    )


total = collections.Counter()

for c in h2:
    total.update(c)

print(
    "\nHasil dengan combiner sama persis:",
    dict(total) == hasil_semua
)

CPU tersedia: 2
skenario                     waktu (s)    pasangan dikirim
1 proses, tanpa combiner         0.549             825,601
1 proses, dengan combiner        1.058              74,009
2 proses, tanpa combiner         1.187             825,601
2 proses, dengan combiner        0.680              74,009

Hasil dengan combiner sama persis: True


Sel ini menguji empat skenario: satu proses tanpa combiner, satu proses dengan combiner, dua proses tanpa combiner, dan dua proses dengan combiner. Setiap skenario dijalankan tiga kali dan menggunakan waktu tengah setelah diurutkan sebagai hasil pengukuran. Contoh modul menunjukkan bahwa combiner mengurangi pasangan dari 825.601 menjadi 74.009, sedangkan waktu aktual bergantung pada komputer.

## 28. H-11.1 — Membuat partitioner dan menghitung beban reducer

In [ ]:
R = 4  # jumlah reducer


def partitioner(kata, r=R):
    return zlib.crc32(
        kata.encode()
    ) % r


def beban(pasangan_per_mapper, r=R):
    b = [0] * r

    for h in pasangan_per_mapper:
        for kata, n in h:
            b[partitioner(kata, r)] += 1

    return b


def tampil(nama, b):
    tot = sum(b)

    print(
        f"{nama:<22}",
        " ".join(
            f"R{i}={x:>7,} ({x/tot*100:4.1f}%)"
            for i, x in enumerate(b)
        )
    )


tampil(
    "tanpa combiner",
    beban(hasil_map)
)

tampil(
    "dengan combiner",
    beban([
        list(c.items())
        for c in h2
    ])
)

tanpa combiner         R0=240,836 (29.2%) R1=170,066 (20.6%) R2=190,727 (23.1%) R3=223,972 (27.1%)
dengan combiner        R0= 18,301 (24.7%) R1= 18,267 (24.7%) R2= 18,530 (25.0%) R3= 18,911 (25.6%)


Kode ini menentukan reducer tujuan sebuah kata menggunakan nilai CRC32 yang dibagi dengan jumlah reducer. Fungsi beban() menghitung banyaknya pasangan yang diterima setiap reducer, sedangkan tampil() menampilkan jumlah dan persentase bebannya. Hasilnya digunakan untuk melihat apakah distribusi pasangan data merata atau mengalami data skew.

## 29. H-11.2 — Membandingkan jumlah reducer

In [ ]:
print(
    "\nBeban reducer terberat dibanding bagian yang adil "
    "(tanpa combiner):"
)

for r in (4, 16, 64):
    b = beban(hasil_map, r)

    print(
        f" {r:>3} reducer: "
        f"terberat {max(b)/sum(b)*100:5.2f}% "
        f"| bagian adil {100/r:5.2f}%"
    )


Beban reducer terberat dibanding bagian yang adil (tanpa combiner):
   4 reducer: terberat 29.17% | bagian adil 25.00%
  16 reducer: terberat  8.82% | bagian adil  6.25%
  64 reducer: terberat  4.28% | bagian adil  1.56%


Sel ini membandingkan persentase beban reducer terberat dengan persentase ideal apabila data terbagi rata. Pengujian dilakukan menggunakan 4, 16, dan 64 reducer untuk melihat apakah penambahan reducer dapat mengurangi ketimpangan beban dalam pemrosesan kata.

## 30. H-11.3 — Memeriksa kata dengan frekuensi tertinggi

In [ ]:
print("\nBeban satu key (tanpa combiner):")

tot = sum(hasil_semua.values())

for kata, n in collections.Counter(
    hasil_semua
).most_common(5):
    print(
        f" {kata:<8} {n:>7,} pasangan "
        f"= {n/tot*100:4.1f}% dari total, "
        f"di reducer R{partitioner(kata)}"
    )


Beban satu key (tanpa combiner):
 di        27,390 pasangan =  3.3% dari total, di reducer R0
 corona    17,239 pasangan =  2.1% dari total, di reducer R2
 ini       10,124 pasangan =  1.2% dari total, di reducer R0
 yang       8,668 pasangan =  1.0% dari total, di reducer R3
 dan        6,785 pasangan =  0.8% dari total, di reducer R3


Kode ini menampilkan lima kata dengan frekuensi tertinggi, jumlah pasangan masing-masing, persentase terhadap total pasangan kata, dan reducer yang menerima setiap kata. Pemeriksaan ini membantu menjelaskan bahwa satu kata yang sangat sering muncul dapat menambah beban reducer tertentu, meskipun jumlah reducer diperbanyak.

## 31. H-12.1 — Membuat file input, mapper, dan reducer

In [ ]:
Path("masukan.tsv").write_text(
    "\n".join(
        f"{k}\t{j}"
        for k, j in zip(
            bersih["kategori"],
            bersih["judul"]
        )
    ) + "\n",
    encoding="utf-8"
)

Path("mapper.py").write_text('''import sys, re

for baris in sys.stdin:
    kategori, judul = baris.rstrip("\\n").split("\\t", 1)

    for kata in re.findall(r"[a-z]+", judul.lower()):
        print(f"{kata}\\t1")
''')

Path("reducer.py").write_text('''import sys

kunci_sekarang, total = None, 0

for baris in sys.stdin:
    kunci, nilai = baris.rstrip("\\n").split("\\t")

    if kunci != kunci_sekarang:
        if kunci_sekarang is not None:
            print(f"{kunci_sekarang}\\t{total}")

        kunci_sekarang, total = kunci, 0

    total += int(nilai)

if kunci_sekarang is not None:
    print(f"{kunci_sekarang}\\t{total}")
''')

print(
    "Baris masukan:",
    sum(1 for _ in open("masukan.tsv", encoding="utf-8"))
)

Baris masukan: 90818


Kode ini menyiapkan file masukan bertipe TSV (tab-separated values), lalu menulis dua skrip terpisah. mapper.py membaca setiap judul dan menghasilkan pasangan kata serta nilai 1. reducer.py menjumlahkan nilai untuk setiap kata yang sama, dengan asumsi masukan yang diterimanya sudah diurutkan berdasarkan key.

## 32. H-12.2 — Menguji mapper dan menjalankan pipeline melalui pipe Linux

In [ ]:
!head -n 3 masukan.tsv | python3 mapper.py

kemnaker	1
awasi	1
tka	1
di	1
meikarta	1
bni	1
digitalkan	1
bni	1
java	1
jazz	1
terbang	1
ke	1
australia	1
edhy	1
prabowo	1
mau	1
genjot	1
budi	1
daya	1
lobster	1


Perintah ini menguji mapper menggunakan tiga baris pertama dari file masukan. Tujuannya untuk memastikan bahwa skrip dapat membaca data TSV dan menghasilkan pasangan kata dengan nilai 1 sebelum dipakai memproses seluruh dataset.

In [ ]:
!time (cat masukan.tsv | python3 mapper.py | LC_ALL=C sort | python3 reducer.py > hasil_pipe.txt)
!wc -l hasil_pipe.txt
!sort -t$'\t' -k2,2nr hasil_pipe.txt | head -n 5


real	0m2.429s
user	0m2.194s
sys	0m0.098s
32275 hasil_pipe.txt
di	27390
corona	17239
ini	10124
yang	8668
dan	6785


Perintah ini menjalankan seluruh pipeline menggunakan pipe Linux. Hasil dari mapper diurutkan menggunakan sort sebagai simulasi tahap shuffle, lalu diteruskan ke reducer. Hasil akhir disimpan di hasil_pipe.txt, jumlah kata unik dihitung, dan lima kata dengan frekuensi tertinggi ditampilkan.

## 33. H-12.3 — Memastikan hasilnya sama dengan H-9

In [ ]:
hasil_pipe = {}

for baris in open("hasil_pipe.txt", encoding="utf-8"):
    kata, n = baris.rstrip("\n").split("\t")
    hasil_pipe[kata] = int(n)

print(
    "Hasil pipe sama dengan hasil H-9:",
    hasil_pipe == hasil_semua
)

Hasil pipe sama dengan hasil H-9: True


Kode ini membaca hasil dari hasil_pipe.txt dan membandingkan frekuensi kata dengan hasil MapReduce pada H-9. Jika output menunjukkan True, kedua cara menghasilkan frekuensi kata yang sama.

## 35. H-13.1 — Menghitung frekuensi kata setiap kategori

In [ ]:
def mapper_kategori(path):
    """Key gabungan (kategori, kata); combiner di mapper."""
    df_ = pd.read_parquet(
        path,
        columns=["judul"]
    )

    kategori = path.parent.name.split("=")[1]
    c = collections.Counter()

    for t in df_["judul"]:
        for kata in tokenisasi(t):
            if kata not in STOPWORD:
                c[(kategori, kata)] += 1

    return c


gabung = collections.Counter()

for c in map(mapper_kategori, splits):
    gabung.update(c)


per_kategori = collections.defaultdict(list)

for (kat, kata), n in gabung.items():
    per_kategori[kat].append((n, kata))


for kat in sorted(per_kategori):
    top5 = sorted(
        per_kategori[kat],
        reverse=True
    )[:5]

    print(
        f"{kat:<8}",
        ", ".join(
            f"{k} ({n:,})"
            for n, k in top5
        )
    )

finance  corona (2,470), rp (1,385), ri (1,125), harga (701), jokowi (642)
food     makanan (699), makan (539), foto (458), galeri (438), enak (358)
health   corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)
hot      corona (1,658), anak (793), rumah (561), meninggal (545), the (528)
inet     corona (845), virus (341), ponsel (309), bikin (253), indonesia (249)
news     corona (7,539), polisi (2,244), kasus (2,015), warga (1,972), positif (1,876)
oto      mobil (1,205), corona (576), motor (432), rp (267), indonesia (256)
sport    corona (228), indonesia (223), motogp (209), masters (127), olimpiade (124)
travel   corona (1,023), wisata (525), foto (333), dunia (299), turis (284)


Kode ini menghitung frekuensi kata berdasarkan gabungan kategori dan kata, dengan stopword diabaikan. Setiap mapper menggunakan Counter untuk menjumlahkan kata secara lokal sebelum hasilnya digabungkan. Setelah semua hasil dikumpulkan, program menampilkan lima kata teratas untuk setiap kategori berita.

36. H-13.2 — Menyimpan hasil ke Parquet dan mengekspor ke SQLite

In [ ]:
import sqlite3

SIAP = Path("siap_pakai")
SIAP.mkdir(exist_ok=True)

baris = [
    (kat, kata, n)
    for (kat, kata), n in gabung.items()
]

hasil_df = pd.DataFrame(
    baris,
    columns=["kategori", "kata", "jumlah"]
)

hasil_df.to_parquet(
    SIAP / "kata_per_kategori.parquet",
    index=False
)


# Simulasi "sqoop export" ke database relasional.
db = sqlite3.connect(
    SIAP / "laporan.db"
)

hasil_df.to_sql(
    "kata_per_kategori",
    db,
    if_exists="replace",
    index=False
)

print(
    "Baris diekspor:",
    db.execute(
        "SELECT COUNT(*) FROM kata_per_kategori"
    ).fetchone()[0]
)

print(
    db.execute("""
        SELECT kata, jumlah
        FROM kata_per_kategori
        WHERE kategori = 'health'
        ORDER BY jumlah DESC
        LIMIT 3
    """).fetchall()
)

db.close()

Baris diekspor: 73601
[('corona', 2590), ('virus', 1040), ('pasien', 471)]


Kode ini menyimpan hasil analisis ke folder siap_pakai dalam format Parquet dan database SQLite. Data dimasukkan ke tabel kata_per_kategori, kemudian query SQL digunakan untuk menghitung jumlah baris yang diekspor dan mengambil tiga kata dengan frekuensi tertinggi dari kategori health. Proses tersebut menyimulasikan fungsi ekspor data ke database seperti pada Sqoop.

# **B. Latihan**

## 1. Stopword domain. Tambahkan 5 kata ke STOPWORD yang menurut Anda mengganggu hasil, lalu jalankan ulang H-13. Catat kata yang masuk dan perubahan lima kata teratas tiap kategori.

In [ ]:
# Simpan stopword asli dari H-9.
STOPWORD_ASLI = set("""
yang di dan ke dari ini itu untuk dengan pada adalah akan tak tidak jadi soal
saat ada bisa juga oleh atau karena sebagai dalam usai hingga jika saja para lebih hanya
sudah belum masih kini lagi pun serta agar bakal dua tiga kok nih ya yg dgn
""".split())


def top5_per_kategori(counter):
    hasil = collections.defaultdict(list)

    for (kategori, kata), jumlah in counter.items():
        hasil[kategori].append((jumlah, kata))

    return {
        kategori: sorted(
            daftar, reverse=True
        )[:5]
        for kategori, daftar in hasil.items()
    }


# Hasil awal H-13 sebelum penambahan stopword.
top5_awal = top5_per_kategori(gabung)

# Tambahkan lima stopword domain.
STOPWORD = set(STOPWORD_ASLI)
STOPWORD.update({"rp", "ri", "the", "foto", "galeri"})

print("Stopword tambahan:", sorted(
    STOPWORD - STOPWORD_ASLI
))

# Jalankan ulang perhitungan H-13.
gabung_baru = collections.Counter()

for hasil_mapper in map(mapper_kategori, splits):
    gabung_baru.update(hasil_mapper)

top5_baru = top5_per_kategori(gabung_baru)

print("\nPerbandingan 5 kata teratas setiap kategori:")

for kategori in sorted(top5_awal):
    awal = top5_awal[kategori]
    baru = top5_baru.get(kategori, [])

    print(f"\nKategori: {kategori}")
    print("Sebelum:", ", ".join(
        f"{kata} ({jumlah:,})"
        for jumlah, kata in awal
    ))
    print("Sesudah:", ", ".join(
        f"{kata} ({jumlah:,})"
        for jumlah, kata in baru
    ))

# Kembalikan stopword asli untuk latihan berikutnya.
STOPWORD = set(STOPWORD_ASLI)

Stopword tambahan: ['foto', 'galeri', 'ri', 'rp', 'the']

Perbandingan 5 kata teratas setiap kategori:

Kategori: finance
Sebelum: corona (2,470), rp (1,385), ri (1,125), harga (701), jokowi (642)
Sesudah: corona (2,470), harga (701), jokowi (642), mau (562), ekonomi (536)

Kategori: food
Sebelum: makanan (699), makan (539), foto (458), galeri (438), enak (358)
Sesudah: makanan (699), makan (539), enak (358), bikin (343), rumah (311)

Kategori: health
Sebelum: corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)
Sesudah: corona (2,590), virus (1,040), pasien (471), indonesia (427), meninggal (413)

Kategori: hot
Sebelum: corona (1,658), anak (793), rumah (561), meninggal (545), the (528)
Sesudah: corona (1,658), anak (793), rumah (561), meninggal (545), film (489)

Kategori: inet
Sebelum: corona (845), virus (341), ponsel (309), bikin (253), indonesia (249)
Sesudah: corona (845), virus (341), ponsel (309), bikin (253), indonesia (249)

Kategori: news
Sebelum: c

Kode menghitung kembali frekuensi kata per kategori setelah lima kata tersebut ditambahkan ke daftar stopword. Hasil sebelum dan sesudah dibandingkan untuk mengetahui kata mana yang hilang dari lima besar dan kata mana yang menggantikannya. Stopword dikembalikan ke daftar awal pada akhir kode agar tidak memengaruhi latihan selanjutnya.

## 2. Per bulan. Ubah key di H-13 menjadi (bulan, kata). Kata apa yang berubah dari Januari ke Mei?

In [ ]:
STOPWORD = set(STOPWORD_ASLI)


def mapper_bulan(path):
    df_ = pd.read_parquet(
        path,
        columns=["judul", "bulan"]
    )

    counter = collections.Counter()

    for bulan, judul in zip(df_["bulan"], df_["judul"]):
        for kata in tokenisasi(judul):
            if kata not in STOPWORD:
                counter[(str(bulan), kata)] += 1

    return counter


# Gabungkan hasil setiap mapper.
gabung_bulan = collections.Counter()

for hasil_mapper in map(mapper_bulan, splits):
    gabung_bulan.update(hasil_mapper)


# Urutkan frekuensi kata untuk setiap bulan.
kata_per_bulan = collections.defaultdict(list)

for (bulan, kata), jumlah in gabung_bulan.items():
    kata_per_bulan[bulan].append((jumlah, kata))


def tampilkan_top_bulanan(bulan, n=5):
    daftar = sorted(
        kata_per_bulan[bulan],
        reverse=True
    )[:n]

    print(f"\nTop {n} kata {bulan}:")

    for jumlah, kata in daftar:
        print(f"{kata:<15} {jumlah:>8,}")

    return {kata for jumlah, kata in daftar}


top_januari = tampilkan_top_bulanan("2020-01")
top_mei = tampilkan_top_bulanan("2020-05")

print("\nKata yang hanya muncul di Top 5 Januari:")
print(sorted(top_januari - top_mei))

print("\nKata yang hanya muncul di Top 5 Mei:")
print(sorted(top_mei - top_januari))


Top 5 kata 2020-01:
banjir             1,220
rp                   616
polisi               554
jokowi               516
tahun                493

Top 5 kata 2020-05:
corona             2,935
kasus                640
positif              628
pandemi              596
warga                565

Kata yang hanya muncul di Top 5 Januari:
['banjir', 'jokowi', 'polisi', 'rp', 'tahun']

Kata yang hanya muncul di Top 5 Mei:
['corona', 'kasus', 'pandemi', 'positif', 'warga']


Pada latihan ini, key MapReduce bukan lagi (kategori, kata), melainkan (bulan, kata). Dengan demikian, frekuensi kata dapat dihitung per bulan. Kode menampilkan lima kata teratas pada Januari dan Mei 2020 serta membandingkan keduanya untuk mengetahui kata yang keluar atau masuk ke dalam daftar lima besar.

## 3. Bigram. Ubah mapper agar menghasilkan pasangan kata berurutan, misalnya "virus corona". Berapa pasangan yang dihasilkan, dan berapa setelah combiner?

In [ ]:
def mapper_bigram(path):
    df_ = pd.read_parquet(
        path,
        columns=["judul"]
    )

    hasil = []

    for judul in df_["judul"]:
        kata = tokenisasi(judul)

        for i in range(len(kata) - 1):
            bigram = f"{kata[i]} {kata[i + 1]}"
            hasil.append((bigram, 1))

    return hasil


# Mapper tanpa combiner.
hasil_map_bigram = [
    mapper_bigram(p)
    for p in splits
]

jumlah_sebelum = sum(
    len(hasil)
    for hasil in hasil_map_bigram
)


# Combiner: jumlahkan bigram pada setiap split.
def combiner_bigram(pasangan):
    counter = collections.Counter()

    for bigram, jumlah in pasangan:
        counter[bigram] += jumlah

    return counter


hasil_combiner_bigram = [
    combiner_bigram(hasil)
    for hasil in hasil_map_bigram
]

jumlah_sesudah = sum(
    len(counter)
    for counter in hasil_combiner_bigram
)


# Reduce: gabungkan hasil semua mapper.
frekuensi_bigram = collections.Counter()

for counter in hasil_combiner_bigram:
    frekuensi_bigram.update(counter)


print("Pasangan bigram sebelum combiner:", f"{jumlah_sebelum:,}")
print("Pasangan bigram setelah combiner:", f"{jumlah_sesudah:,}")
print("Jumlah bigram unik:", f"{len(frekuensi_bigram):,}")

print("\n10 bigram teratas:")

for bigram, jumlah in frekuensi_bigram.most_common(10):
    print(f"{bigram:<25} {jumlah:>8,}")

print(
    "\nTotal frekuensi tetap sama:",
    sum(frekuensi_bigram.values()) == jumlah_sebelum
)

Pasangan bigram sebelum combiner: 734,783
Pasangan bigram setelah combiner: 411,217
Jumlah bigram unik: 349,114

10 bigram teratas:
virus corona                 2,915
corona di                    2,178
positif corona               1,804
di tengah                    1,258
pandemi corona               1,075
di rumah                     1,047
new normal                     917
di indonesia                   871
rp juta                        700
hari ini                       627

Total frekuensi tetap sama: True


Bigram dibentuk dari dua kata berurutan pada setiap judul berita. Mapper menghasilkan satu pasangan untuk setiap bigram, sedangkan combiner menjumlahkan bigram yang sama di dalam setiap split sebelum dikirim ke tahap reduce. Jumlah pasangan yang dikirim seharusnya berkurang, tetapi total frekuensi bigram setelah reduce tetap sama dengan jumlah sebelum combiner.

## 4. Jumlah reducer. Jalankan H-11 dengan R = 2, 8, dan 32. Buat tabel beban reducer terberat dibanding bagian yang adil.

In [ ]:
# Menggunakan hasil_map dari H-9 dan fungsi beban() dari H-11.
daftar_reducer = [2, 8, 32]
hasil_perbandingan = []

for r in daftar_reducer:
    beban_reducer = beban(hasil_map, r)

    total = sum(beban_reducer)
    terberat = max(beban_reducer) / total * 100
    ideal = 100 / r

    hasil_perbandingan.append({
        "Jumlah reducer": r,
        "Reducer terberat (%)": round(terberat, 2),
        "Bagian ideal (%)": round(ideal, 2),
        "Selisih (%)": round(terberat - ideal, 2)
    })

tabel_reducer = pd.DataFrame(hasil_perbandingan)

print(tabel_reducer.to_string(index=False))

 Jumlah reducer  Reducer terberat (%)  Bagian ideal (%)  Selisih (%)
              2                 52.27             50.00         2.27
              8                 16.83             12.50         4.33
             32                  5.82              3.12         2.70


Kode mengukur persentase pasangan data yang diterima reducer dengan beban tertinggi untuk tiga konfigurasi. Persentase ideal dihitung dengan rumus \(100/R\), yaitu 50% untuk dua reducer, 12,5% untuk delapan reducer, dan 3,125% untuk 32 reducer. Selisih yang semakin besar menunjukkan pembagian beban yang semakin jauh dari kondisi ideal. Jumlah reducer yang bertambah tidak menjamin beban terbagi rata karena distribusi key dan frekuensinya turut memengaruhi hasil.

## 5. Crawl ulang. Tambahkan jeda 0,05 detik pada crawl_iter untuk 50 halaman pertama. Berapa waktunya, dan berapa lama bila 911 halaman memakai jeda 1 detik?

In [ ]:
import time

JUMLAH_UJI = 50
JEDA_UJI = 0.05
JUMLAH_HALAMAN = 911
JEDA_PENUH = 1.0

t0 = time.perf_counter()

hasil_crawl_uji = list(
    crawl_iter(
        BASE,
        range(1, JUMLAH_UJI + 1),
        delay=JEDA_UJI
    )
)

waktu_50 = time.perf_counter() - t0

# Perkirakan waktu pemrosesan per halaman,
# dengan mengurangi waktu tidur yang ditambahkan.
waktu_tidur_uji = JUMLAH_UJI * JEDA_UJI

waktu_kerja_per_halaman = max(
    0,
    waktu_50 - waktu_tidur_uji
) / JUMLAH_UJI

estimasi_911 = (
    waktu_kerja_per_halaman + JEDA_PENUH
) * JUMLAH_HALAMAN

print("Jumlah judul dari 50 halaman:", len(hasil_crawl_uji))
print("Waktu crawling 50 halaman:", round(waktu_50, 2), "detik")
print("Estimasi waktu 911 halaman:", round(estimasi_911, 2), "detik")
print("Estimasi dalam menit:", round(estimasi_911 / 60, 2), "menit")

Jumlah judul dari 50 halaman: 5000
Waktu crawling 50 halaman: 5.65 detik
Estimasi waktu 911 halaman: 968.34 detik
Estimasi dalam menit: 16.14 menit


Kode mengukur waktu crawling 50 halaman dengan jeda 0,05 detik, kemudian memperkirakan durasi crawling 911 halaman dengan jeda satu detik per halaman. Perkiraan ini menggunakan waktu rata-rata pemrosesan setiap halaman dari pengujian 50 halaman, di luar waktu tidur yang ditambahkan. Berdasarkan contoh modul, crawling 911 halaman tanpa jeda memerlukan sekitar 47 detik; jika kinerja serupa dan ditambah jeda satu detik per halaman, perkiraan kasarnya menjadi sekitar 958 detik atau 15 menit 58 detik. Waktu aktual dapat berbeda karena jaringan dan kondisi runtime.

## 6. Sitemap lain. Ganti satu alamat di SITEMAP_DIPILIH dengan sitemap_news dari bagian lain yang tercantum di sitemap.xml. Kategori apa yang muncul? Terapkan aturan_kualitas dari H-6 pada 20 judul hasil crawl dan laporkan hasilnya.

In [ ]:
URL_INDEKS = "https://www.detik.com/sitemap.xml"

respons = requests.get(
    URL_INDEKS,
    headers={"User-Agent": UA_DETIK},
    timeout=20
)

respons.raise_for_status()

sitemap_index = BeautifulSoup(respons.content, "xml")

daftar_sitemap = [
    loc.get_text(strip=True)
    for loc in sitemap_index.find_all("loc")
    if "sitemap_news.xml" in loc.get_text()
]

alternatif = [
    url for url in daftar_sitemap
    if url not in SITEMAP_DIPILIH
]

print("Jumlah sitemap berita dalam indeks:", len(daftar_sitemap))
print("\nContoh sitemap alternatif:")

for url in alternatif[:10]:
    print(url)

Jumlah sitemap berita dalam indeks: 248

Contoh sitemap alternatif:
https://news.detik.com/suara-pembaca/sitemap_news.xml
https://news.detik.com/internasional/sitemap_news.xml
https://news.detik.com/bbc-world/sitemap_news.xml
https://news.detik.com/abc-australia/sitemap_news.xml
https://news.detik.com/dw/sitemap_news.xml
https://news.detik.com/kolom/sitemap_news.xml
https://news.detik.com/prokontra/sitemap_news.xml
https://news.detik.com/pemilu/sitemap_news.xml
https://news.detik.com/pilkada/sitemap_news.xml
https://news.detik.com/melindungi-tuah-marwah/sitemap_news.xml


Kode membaca indeks sitemap dan memilih alamat yang mengandung sitemap_news.xml. Alamat yang sudah digunakan pada H-4 dikeluarkan dari daftar sehingga tersisa sitemap alternatif. Apabila URL indeks tersebut tidak memberikan XML indeks yang sesuai pada runtime, gunakan URL indeks yang benar dari situs sesuai petunjuk modul.

In [ ]:
if not alternatif:
    raise ValueError("Tidak ditemukan sitemap berita alternatif.")

# Ganti sitemap ketiga dengan sitemap alternatif.
sitemap_lama = SITEMAP_DIPILIH[2]
SITEMAP_DIPILIH[2] = alternatif[0]

print("Sitemap lama:", sitemap_lama)
print("Sitemap baru:", SITEMAP_DIPILIH[2])

try:
    hasil_crawl_alternatif = crawl_detik()
except Exception as e:
    hasil_crawl_alternatif = []
    print("Crawling gagal:", type(e).__name__, "-", e)

if hasil_crawl_alternatif:
    df_alternatif = pd.DataFrame(hasil_crawl_alternatif)

    print("\nJumlah judul:", len(df_alternatif))
    print("\nDistribusi kategori:")
    print(df_alternatif["kategori"].value_counts(dropna=False))

    print("\nContoh judul:")
    print(
        df_alternatif[
            ["judul", "kategori", "tanggal"]
        ].head().to_string(index=False)
    )

Sitemap lama: https://sport.detik.com/basket/sitemap_news.xml
Sitemap baru: https://news.detik.com/suara-pembaca/sitemap_news.xml

Jumlah judul: 20

Distribusi kategori:
kategori
news    13
inet     7
Name: count, dtype: int64

Contoh judul:
                                                                        judul kategori    tanggal
Andre Rosiade Cek MBG di Padang: Alhamdulillah Anak Senang, Ingin Dilanjutkan     news 2026-10-29
   Kemenkop-Pertamina Luncurkan Model Pengumpulan Minyak Jelantah Lewat KDKMP     news 2026-10-29
    Rumah-Mobil Disita, Anggota DPRD Vinna Ledy No Comment Usai Diperiksa KPK     news 2026-10-29
    Kemenkop-Pertamina Resmikan Piloting Pengumpulan Minyak Jelantah di KDKMP     news 2026-10-29
         Cucu Dibakar Nenek di Depok karena Tak Dijemput Alami Luka Bakar 80%     news 2026-10-29


Sel ini mengganti satu alamat sitemap dan menjalankan kembali crawl_detik(). Kode kemudian menampilkan jumlah judul, kategori yang diperoleh, serta contoh hasilnya. Kategori yang muncul ditentukan oleh sitemap alternatif yang terpilih, sehingga hasilnya tidak selalu sama setiap kali dijalankan.

In [ ]:
if hasil_crawl_alternatif:
    for nama, lulus in aturan_kualitas(df_alternatif).items():
        gagal = (~lulus).sum()
        persen = (~lulus).mean() * 100

        print(
            f"{nama:<24} gagal: {gagal:>3} "
            f"({persen:5.2f}%)"
        )
else:
    print("Tidak ada data crawling yang dapat diperiksa.")

judul tidak kosong       gagal:   0 ( 0.00%)
judul 3-30 kata          gagal:   0 ( 0.00%)
kategori dikenal         gagal:   0 ( 0.00%)
tanggal valid            gagal:   0 ( 0.00%)
url artikel valid        gagal:   0 ( 0.00%)
judul tidak duplikat     gagal:   0 ( 0.00%)


Kode menerapkan fungsi aturan_kualitas() dari H-6 untuk memeriksa judul kosong, jumlah kata, kategori, tanggal, URL, dan duplikasi. Hasilnya menampilkan jumlah baris yang gagal serta persentasenya untuk setiap aturan, sehingga dapat digunakan dalam laporan kualitas data dari crawling alternatif.

## 7. Kata kunci. Kolom kata_kunci berisi kata kunci tiap berita (dipisah koma). Tulis mapper dan reducer untuk menghitung kata kunci terbanyak pada hasil crawl Anda.

In [ ]:
# Gunakan hasil crawling alternatif dari nomor 6.
# Jika tidak tersedia, gunakan hasil crawling awal dari H-4.
data_kata_kunci = (
    hasil_crawl_alternatif
    if hasil_crawl_alternatif
    else sampel
)


def mapper_kata_kunci(berita):
    """Menghasilkan pasangan (kata_kunci, 1)."""
    raw = berita.get("kata_kunci", "")

    if not isinstance(raw, str):
        return []

    hasil = []

    for kata in raw.split(","):
        kata = kata.strip().lower()

        if kata:
            hasil.append((kata, 1))

    return hasil


# MAP
hasil_map_keyword = [
    mapper_kata_kunci(berita)
    for berita in data_kata_kunci
]

# SHUFFLE
shuffle_keyword = collections.defaultdict(list)

for hasil_mapper in hasil_map_keyword:
    for kata, jumlah in hasil_mapper:
        shuffle_keyword[kata].append(jumlah)

# REDUCE
frekuensi_keyword = {
    kata: sum(nilai)
    for kata, nilai in shuffle_keyword.items()
}

print("Jumlah berita:", len(data_kata_kunci))
print("Jumlah kata kunci unik:", len(frekuensi_keyword))

print("\n10 kata kunci teratas:")

for kata, jumlah in sorted(
    frekuensi_keyword.items(),
    key=lambda item: item[1],
    reverse=True
)[:10]:
    print(f"{kata:<30} {jumlah:>5}")

Jumlah berita: 20
Jumlah kata kunci unik: 63

10 kata kunci teratas:
suara pembaca                      6
jabodetabek                        3
bnr                                2
minyak jelantah                    2
kemenkop                           2
kecerdasan buatan                  2
samsung solve for tomorrow         2
andre rosiade                      1
makan bergizi gratis               1
mbg                                1


Fungsi mapper_kata_kunci() memecah kolom kata_kunci berdasarkan tanda koma, kemudian menghasilkan pasangan (kata_kunci, 1). Tahap shuffle mengelompokkan kata kunci yang sama, sedangkan reducer menjumlahkan frekuensi kemunculannya. Hasil akhir berupa daftar kata kunci yang paling sering muncul pada berita yang berhasil di-crawl.